In [5]:
import pandas as pd
import datetime as dt

# 1.
df = pd.read_csv(r"C:\Users\nithish\OneDrive\Documents\Task3(Apex)\Sample - Superstore.csv", encoding='latin1')

# 2.
df['Order Date'] = pd.to_datetime(df['Order Date'], format='mixed', errors='coerce')

# 3.
df = df.dropna(subset=['Order Date'])

# 4.
snapshot_date = df['Order Date'].max() + dt.timedelta(days=1)
print(f"Analysis Snapshot Date: {snapshot_date}")

Analysis Snapshot Date: 2017-12-31 00:00:00


In [6]:
# Grouping by Customer Name
rfm = df.groupby('Customer Name').agg({
    'Order Date': lambda x: (snapshot_date - x.max()).days,  # Recency
    'Order ID': 'nunique',                                   # Frequency
    'Sales': 'sum'                                           # Monetary
})

# Renaming columns for clarity
rfm.rename(columns={
    'Order Date': 'Recency',
    'Order ID': 'Frequency',
    'Sales': 'Monetary'
}, inplace=True)

print(rfm.head())

                 Recency  Frequency  Monetary
Customer Name                                
Aaron Bergman        416          3   886.156
Aaron Hawkins         13          7  1744.700
Aaron Smayling        89          7  3050.692
Adam Bellavance       55          8  7755.620
Adam Hart             35         10  3250.337


In [7]:
# Dividing customers into 4 equal groups based on scores (1 is worst, 4 is best)
rfm['R_Score'] = pd.qcut(rfm['Recency'], q=4, labels=[4, 3, 2, 1]) # Lower recency days = better score
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), q=4, labels=[1, 2, 3, 4])
rfm['M_Score'] = pd.qcut(rfm['Monetary'], q=4, labels=[1, 2, 3, 4])

# Combining scores to build a segmentation logic
def assign_segment(row):
    r, f, m = row['R_Score'], row['F_Score'], row['M_Score']
    if r >= 3 and f >= 3 and m >= 3:
        return 'Champions'
    elif r <= 2 and f >= 3:
        return 'At Risk / Loyalists Slipping'
    elif r >= 3 and f <= 2:
        return 'New / Recent Buyers'
    else:
        return 'Lost Customers'

rfm['Customer_Segment'] = rfm.apply(assign_segment, axis=1)


rfm.to_csv('Superstore_Customer_Segments.csv')
print(rfm['Customer_Segment'].value_counts())

Customer_Segment
Lost Customers                  300
Champions                       176
New / Recent Buyers             159
At Risk / Loyalists Slipping    158
Name: count, dtype: int64
